In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as sp
from scipy.optimize import minimize_scalar
import qutip as qt
from tqdm.auto import tqdm

In [ ]:
# System Parameters for Single Symbol Simulation
alpha = 2.0             # Coherent state amplitude (|alpha_0> = |+alpha>, |alpha_1> = |-alpha>)
alpha0 = alpha          # Hypothesis 0 amplitude
alpha1 = -alpha         # Hypothesis 1 amplitude
nu = 2.0                # Dark count rate (counts per second)
T = 1.0                 # Symbol duration (seconds)
dt = 1e-4               # Time step resolution
prior_P0 = 0.5          # Initial prior probability for state |alpha_0>
steps = int(T / dt)


In [ ]:
true_state_idx = 1
true_alpha = alpha0 if true_state_idx == 0 else alpha1

# Initialize tracking variables
P0 = prior_P0
P0_history = np.zeros(steps)
beta_history = np.zeros(steps)
clicks = []

# Continuous time-slice simulation of the optimal Dolinar receiver with off-center displacement
for i in tqdm(range(steps), desc="Simulating Single Symbol"):
    t = i * dt
    P0_history[i] = P0
    
    # 1. Optimal Dolinar off-center displacement field beta(t)
    # Time-to-go offset factor: f(t) = exp(-4 * alpha^2 * (T - t))
    # beta(t) = -alpha * [ (2*P0 - 1) + sqrt(1 - f(t)) ] / [ 1 + (2*P0 - 1)*sqrt(1 - f(t)) ]
    f_t = np.exp(-4.0 * alpha**2 * (T - t))
    denom = np.sqrt(np.maximum(1.0 - f_t, 1e-12))
    x = 2.0 * P0 - 1.0
    beta = -alpha * (x + denom) / (1.0 + x * denom)
    beta_history[i] = beta
    
    # 2. Instantaneous Poisson click rates (displaced signal + dark counts)
    rate0 = np.abs(alpha0 + beta)**2 + nu
    rate1 = np.abs(alpha1 + beta)**2 + nu
    rate_true = np.abs(true_alpha + beta)**2 + nu
    
    # 3. Exact on/off detector click probability in interval dt
    p_click_true = 1.0 - np.exp(-rate_true * dt)
    
    # 4. Bayesian continuous drift and discrete jump state update
    if np.random.rand() < p_click_true:
        # Detector clicked: discrete jump update
        num = P0 * rate0
        den = num + (1.0 - P0) * rate1
        P0 = num / den if den > 0 else 0.5
        clicks.append(t)
    else:
        # No click: continuous deterministic drift update
        num = P0 * np.exp(-rate0 * dt)
        den = num + (1.0 - P0) * np.exp(-rate1 * dt)
        P0 = num / den if den > 0 else 0.5
        
    P0 = np.clip(P0, 1e-12, 1.0 - 1e-12)

# Final decision based on ending posterior probability
decision = 0 if P0 > 0.5 else 1
print(f"Completed: True state = |alpha_{true_state_idx}>, Receiver Decision = |alpha_{decision}>, Total Clicks = {len(clicks)}")


In [ ]:
time_axis = np.linspace(0, T, steps)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Plot Posterior Probability P0(t)
ax1.plot(time_axis, P0_history, label=r'Posterior Probability $P_0(t)$', color='darkmagenta', linewidth=1.8)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Decision Boundary (0.5)')
for click in clicks:
    ax1.axvline(x=click, color='red', linestyle=':', alpha=0.8)
if clicks:
    ax1.axvline(x=clicks[0], color='red', linestyle=':', alpha=0.8, label='Detector Click')
ax1.set_ylabel(r"$P_0(t)$", fontsize=11)
ax1.set_title(f"Optimal Dolinar Receiver: True State $|\\alpha_{true_state_idx}\\rangle$, Decision $|\\alpha_{decision}\\rangle$ (Dark counts $\\nu={nu}$)")
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot Dynamic Off-Center Displacement Field beta(t)
ax2.plot(time_axis, beta_history, label=r'Off-Center LO Displacement $\beta(t)$', color='teal', linewidth=1.8)
ax2.axhline(-alpha, color='blue', linestyle=':', alpha=0.5, label=r'Fixed Kennedy Nulling ($-\alpha$)')
ax2.axhline(alpha, color='orange', linestyle=':', alpha=0.5, label=r'Fixed Kennedy Nulling ($+\alpha$)')
for click in clicks:
    ax2.axvline(x=click, color='red', linestyle=':', alpha=0.8)
ax2.set_xlabel("Time (s)", fontsize=11)
ax2.set_ylabel(r"$\beta(t)$", fontsize=11)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Theoretical Quantum Error Bounds for BPSK vs Amplitude alpha
def helstrom_bound(alpha, pi0=0.5):
    """Quantum Helstrom minimum error probability bound for BPSK with prior pi0 = P(H0)"""
    pi1 = 1.0 - pi0
    overlap_sq = np.exp(-4.0 * alpha**2)
    return 0.5 * (1.0 - np.sqrt(np.maximum(1.0 - 4.0 * pi0 * pi1 * overlap_sq, 0.0)))

def homodyne_bound(alpha, pi0=0.5):
    """Standard Quantum Limit (SQL) via ideal homodyne detection"""
    pi1 = 1.0 - pi0
    x_th = (1.0 / (4.0 * np.maximum(alpha, 1e-6))) * np.log(pi1 / pi0)
    err0 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha - x_th))
    err1 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha + x_th))
    return pi0 * err0 + pi1 * err1

def standard_kennedy_bound(alpha, pi0=0.5):
    """Standard Kennedy bound (fixed nulling beta = -alpha)"""
    pi1 = 1.0 - pi0
    return pi1 * np.exp(-4.0 * alpha**2)

def get_optimal_kennedy_displacement(alpha, pi0=0.5, nu=0.0, eta=1.0):
    """Finds optimal over-displacement magnitude beta_opt for Kennedy receiver"""
    pi1 = 1.0 - pi0
    def loss(b):
        r0 = eta * (alpha - b)**2 + nu
        r1 = eta * (alpha + b)**2 + nu
        err0 = 1.0 - np.exp(-r0)
        err1 = np.exp(-r1)
        return pi0 * err0 + pi1 * err1
    res = minimize_scalar(loss, bounds=(alpha, alpha + 3.0), method='bounded')
    return res.x

def optimized_kennedy_bound(alpha, pi0=0.5):
    """Displacement-Optimized Kennedy bound (beta = -beta_opt > alpha)"""
    pi1 = 1.0 - pi0
    if np.isscalar(alpha):
        b_opt = get_optimal_kennedy_displacement(alpha, pi0=pi0)
        r0 = (alpha - b_opt)**2
        r1 = (alpha + b_opt)**2
        return pi0 * (1.0 - np.exp(-r0)) + pi1 * np.exp(-r1)
    else:
        return np.array([optimized_kennedy_bound(a, pi0=pi0) for a in alpha])

# 1. Dolinar Receiver with HJB Parity Testing (Geremia 2004 Model - Hits Helstrom Bound)
def simulate_geremia_dolinar(alpha, eta=1.0, pi0=0.5, trials=30000, steps=1000, T=1.0):
    """
    Simulates the Geremia (2004) Dolinar receiver policy for BPSK.
    Achieves the exact Helstrom bound across all alphas.
    """
    pi1 = 1.0 - pi0
    dt = T / steps
    N_bar = 4.0 * alpha**2
    
    true_states = np.random.choice([0, 1], p=[pi0, pi1], size=trials)
    true_signals = np.where(true_states == 0, 0.0, np.sqrt(N_bar))
    
    clicks = np.zeros(trials, dtype=int)
    t_arr = np.linspace(dt, T, steps)
    n_arr = N_bar * (t_arr / T)
    
    sqrt_term = np.sqrt(np.maximum(1.0 - 4.0 * pi0 * pi1 * np.exp(-eta * n_arr), 1e-10))
    J_arr = 0.5 * (1.0 - sqrt_term)
    
    u1_arr = -np.sqrt(N_bar) * (1.0 + J_arr / sqrt_term)
    u0_arr = np.sqrt(N_bar) * (J_arr / sqrt_term)
    
    for k in range(steps):
        u = np.where(clicks % 2 == 0, u1_arr[k], u0_arr[k])
        rate = eta * (true_signals + u)**2
        p_click = 1.0 - np.exp(-rate * dt)
        clicked = np.random.rand(trials) < p_click
        clicks += clicked.astype(int)
        
    decisions = np.where(clicks % 2 == 0, 1, 0)
    return np.mean(decisions != true_states)

# 2. Dolinar Receiver with Real-Time Bayesian Drift
def simulate_bayesian_dolinar(alpha, prior_p0=0.5, nu=0.0, eta=1.0, trials=30000, T=1.0):
    """
    Simulates the Bayesian continuous-drift Dolinar receiver (tracks 0.5 * exp(-4*alpha^2)).
    """
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    decisions = np.zeros(trials, dtype=int)
    
    for i in range(trials):
        s_alpha = true_alphas[i]
        t = 0.0
        P0 = prior_p0
        
        while t < T:
            f_t = np.exp(-4.0 * alpha**2 * (T - t))
            denom = np.sqrt(max(1.0 - f_t, 1e-12))
            x = 2.0 * P0 - 1.0
            beta = -alpha * (x + denom) / (1.0 + x * denom)
            
            rate_signal = np.abs(s_alpha + beta)**2
            rate_total = eta * rate_signal + nu
            
            if rate_total <= 1e-12:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
                
            delta_t = np.random.exponential(1.0 / rate_total)
            t_next = t + delta_t
            
            if t_next >= T:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
            else:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                P0 = P0 * np.exp(-rate0 * delta_t) / (P0 * np.exp(-rate0 * delta_t) + (1.0 - P0) * np.exp(-rate1 * delta_t))
                P0 = (P0 * rate0) / (P0 * rate0 + (1.0 - P0) * rate1)
                t = t_next
                P0 = np.clip(P0, 1e-12, 1.0 - 1e-12)
                
        decisions[i] = 0 if P0 > 0.5 else 1
        
    return np.mean(decisions != true_states)

# 3. Static Kennedy Detection
def simulate_kennedy(alpha, prior_p0=0.5, nu=0.0, eta=1.0, trials=30000, optimize_disp=False, T=1.0):
    """Simulates static Kennedy detection without dynamic feedback."""
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    
    if optimize_disp:
        beta_mag = get_optimal_kennedy_displacement(alpha, pi0=prior_p0, nu=nu, eta=eta)
        beta = -beta_mag
    else:
        beta = -alpha
        
    rate_true = eta * np.abs(true_alphas + beta)**2 + nu
    clicks = np.random.poisson(rate_true * T)
    decisions = np.where(clicks == 0, 0, 1)
    return np.mean(decisions != true_states)

# Sweep Helper
def sweep_ber(alpha_vals, sim_fn, **kwargs):
    desc = kwargs.pop('desc', 'Simulating')
    bers = []
    for a in tqdm(alpha_vals, desc=desc):
        bers.append(sim_fn(a, **kwargs))
    return np.array(bers)

# Configuration Parameters
prior_prob = 0.5          # Prior probability P(H0)
eta = 1.0                 # Detector quantum efficiency
alpha_points = np.linspace(0.1, 1.6, 16)
trials_count = 30000

# Run Simulations
ber_geremia_dolinar = sweep_ber(alpha_points, simulate_geremia_dolinar, eta=eta, pi0=prior_prob, trials=trials_count, desc="Dolinar (Geremia HJB Model)")
ber_bayesian_dolinar = sweep_ber(alpha_points, simulate_bayesian_dolinar, prior_p0=prior_prob, eta=eta, trials=trials_count, desc="Dolinar (Bayesian Adaptive)")
ber_std_kennedy = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, eta=eta, trials=trials_count, optimize_disp=False, desc="Std Kennedy (beta=-alpha)")
ber_opt_kennedy = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, eta=eta, trials=trials_count, optimize_disp=True, desc="Opt Kennedy (beta=-beta_opt)")

# Smooth theoretical curves
alpha_grid = np.linspace(0.05, 1.8, 200)
helstrom_curve = helstrom_bound(alpha_grid, pi0=prior_prob)
homodyne_curve = homodyne_bound(alpha_grid, pi0=prior_prob)
std_kennedy_curve = standard_kennedy_bound(alpha_grid, pi0=prior_prob)
opt_kennedy_curve = optimized_kennedy_bound(alpha_grid, pi0=prior_prob)

# ==============================================================================
# Plotting (BER vs Amplitude alpha)
# ==============================================================================
plt.figure(figsize=(11, 7.5))

# Theoretical Curves
plt.semilogy(alpha_grid, helstrom_curve, 'k-', linewidth=2.5, label='Helstrom Bound (Quantum Limit)')
plt.semilogy(alpha_grid, homodyne_curve, 'g--', linewidth=2.0, label='Homodyne Detection (SQL)')
plt.semilogy(alpha_grid, std_kennedy_curve, 'c-.', linewidth=2.0, label=r'Standard Kennedy Bound ($\\beta=-\\alpha$)')
plt.semilogy(alpha_grid, opt_kennedy_curve, color='darkorange', linestyle=':', linewidth=2.2, label=r'Optimized Kennedy Bound ($\\beta=-\\beta_{opt}$)')

# Simulated Data Points
plt.semilogy(alpha_points, ber_geremia_dolinar, 'g^', markersize=7, label='Dolinar Receiver (Geremia HJB Model)')
plt.semilogy(alpha_points, ber_bayesian_dolinar, 'mo', markersize=6, label='Dolinar Receiver (Bayesian Adaptive)')
plt.semilogy(alpha_points, ber_std_kennedy, 'c+', markersize=8, markeredgewidth=2, label=r'Standard Kennedy ($\\beta=-\\alpha$)')
plt.semilogy(alpha_points, ber_opt_kennedy, 'rx', markersize=7, markeredgewidth=1.8, label=r'Optimized Kennedy ($\\beta=-\\beta_{opt}$)')

plt.title(r'BPSK Quantum Receiver Discrimination: BER vs. Amplitude $\alpha$ ($P_0=' + str(prior_prob) + r'$)', fontsize=12)
plt.xlabel(r'Coherent State Amplitude $\alpha$ (Mean Photon Number $\bar{n}=\alpha^2$)', fontsize=11)
plt.ylabel('Bit Error Rate (BER)', fontsize=11)
plt.ylim(1e-6, 0.8)
plt.xlim(0.1, 1.8)
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.legend(loc='best', fontsize=9.5)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Performance Comparison under Realistic Detector Imperfections
# (1. Ideal, 2. Dark Counts, 3. Subunity Efficiency, 4. Both Combined)
# ==============================================================================

# Simulation Grid and Parameters
prior_prob = 0.5
alpha_pts = np.linspace(0.1, 1.5, 15)
alpha_fine = np.linspace(0.05, 1.6, 200)
mc_trials = 25000

# 4 Regimes of Imperfections
scenarios = [
    {"title": "1. Ideal Detector (\\eta=1.0, \\nu=0.0)", "eta": 1.0, "nu": 0.0},
    {"title": "2. Dark Counts Only (\\eta=1.0, \\nu=0.02)", "eta": 1.0, "nu": 0.02},
    {"title": "3. Subunity Efficiency Only (\\eta=0.65, \\nu=0.0)", "eta": 0.65, "nu": 0.0},
    {"title": "4. Combined Imperfections (\\eta=0.65, \\nu=0.02)", "eta": 0.65, "nu": 0.02}
]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, sc in enumerate(scenarios):
    eta_val = sc["eta"]
    nu_val = sc["nu"]
    ax = axes[idx]
    
    # 1. Theoretical Bounds
    h_bound = helstrom_bound(alpha_fine, eta=eta_val, pi0=prior_prob)
    k_bound = optimized_kennedy_bound(alpha_fine, eta=eta_val, nu=nu_val, pi0=prior_prob)
    
    # 2. Monte Carlo Simulations
    d_sim = [simulate_geremia_dolinar(a, eta=eta_val, nu=nu_val, pi0=prior_prob, trials=mc_trials) 
             for a in tqdm(alpha_pts, desc=f"Dolinar: {sc['title']}")]
    
    k_sim = [simulate_optimized_kennedy(a, eta=eta_val, nu=nu_val, pi0=prior_prob, trials=mc_trials) 
             for a in tqdm(alpha_pts, desc=f"Opt-Kennedy: {sc['title']}")]
    
    # 3. Plotting Subplot
    ax.semilogy(alpha_fine, h_bound, 'k-', linewidth=2.2, label=f'Helstrom Bound (\\eta={eta_val})')
    ax.semilogy(alpha_fine, k_bound, color='darkorange', linestyle=':', linewidth=2.0, label='Optimized Kennedy Bound')
    
    ax.semilogy(alpha_pts, d_sim, 'g^', markersize=7, label='Dolinar Receiver (Geremia HJB)')
    ax.semilogy(alpha_pts, k_sim, 'rx', markersize=7, markeredgewidth=1.8, label='Optimized Kennedy Receiver')
    
    ax.set_title(sc["title"], fontsize=12, fontweight='bold')
    ax.set_xlabel(r'Coherent State Amplitude $\alpha$', fontsize=10.5)
    ax.set_ylabel('Bit Error Rate (BER)', fontsize=10.5)
    ax.set_xlim(0.1, 1.6)
    ax.set_ylim(1e-4, 0.8)
    ax.grid(True, which='both', linestyle=':', alpha=0.5)
    ax.legend(loc='best', fontsize=9)

plt.suptitle('Dolinar vs. Optimized Kennedy Receiver under Realistic Detection Inefficiencies', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import scipy.special as sp
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ==============================================================================
# 1. Theoretical Quantum Error Bounds vs. Amplitude alpha (BPSK)
# ==============================================================================

def helstrom_bound(alpha, pi0=0.5):
    """Quantum Helstrom minimum error probability bound for BPSK"""
    pi1 = 1.0 - pi0
    overlap_sq = np.exp(-4.0 * alpha**2)
    return 0.5 * (1.0 - np.sqrt(np.maximum(1.0 - 4.0 * pi0 * pi1 * overlap_sq, 0.0)))

def homodyne_bound(alpha, pi0=0.5):
    """Standard Quantum Limit (SQL) via ideal homodyne detection"""
    pi1 = 1.0 - pi0
    x_th = (1.0 / (4.0 * np.maximum(alpha, 1e-6))) * np.log(pi1 / pi0)
    err0 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha - x_th))
    err1 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha + x_th))
    return pi0 * err0 + pi1 * err1

def standard_kennedy_bound(alpha, pi0=0.5):
    """Standard Kennedy bound (fixed nulling beta = -alpha)"""
    pi1 = 1.0 - pi0
    return pi1 * np.exp(-4.0 * alpha**2)

def get_optimal_kennedy_displacement(alpha, pi0=0.5, nu=0.0, eta=1.0):
    """Finds optimal over-displacement magnitude beta_opt for Kennedy receiver"""
    pi1 = 1.0 - pi0
    def loss(b):
        r0 = eta * (alpha - b)**2 + nu
        r1 = eta * (alpha + b)**2 + nu
        err0 = 1.0 - np.exp(-r0)
        err1 = np.exp(-r1)
        return pi0 * err0 + pi1 * err1
    res = minimize_scalar(loss, bounds=(alpha, alpha + 3.0), method='bounded')
    return res.x

def optimized_kennedy_bound(alpha, pi0=0.5):
    """Displacement-Optimized Kennedy bound (beta = -beta_opt > alpha)"""
    pi1 = 1.0 - pi0
    if np.isscalar(alpha):
        b_opt = get_optimal_kennedy_displacement(alpha, pi0=pi0)
        r0 = (alpha - b_opt)**2
        r1 = (alpha + b_opt)**2
        return pi0 * (1.0 - np.exp(-r0)) + pi1 * np.exp(-r1)
    else:
        return np.array([optimized_kennedy_bound(a, pi0=pi0) for a in alpha])

# ==============================================================================
# 2. Monte Carlo Simulators
# ==============================================================================

def simulate_geremia_dolinar(alpha, eta=1.0, pi0=0.5, trials=30000, steps=1000, T=1.0):
    """
    Dolinar receiver using Geremia's (2004) HJB optimal control policy.
    Achieves the exact Helstrom bound across all alphas.
    """
    pi1 = 1.0 - pi0
    dt = T / steps
    N_bar = 4.0 * alpha**2
    
    true_states = np.random.choice([0, 1], p=[pi0, pi1], size=trials)
    true_signals = np.where(true_states == 0, 0.0, np.sqrt(N_bar))
    
    clicks = np.zeros(trials, dtype=int)
    t_arr = np.linspace(dt, T, steps)
    n_arr = N_bar * (t_arr / T)
    
    sqrt_term = np.sqrt(np.maximum(1.0 - 4.0 * pi0 * pi1 * np.exp(-eta * n_arr), 1e-10))
    J_arr = 0.5 * (1.0 - sqrt_term)
    
    u1_arr = -np.sqrt(N_bar) * (1.0 + J_arr / sqrt_term)
    u0_arr = np.sqrt(N_bar) * (J_arr / sqrt_term)
    
    for k in range(steps):
        u = np.where(clicks % 2 == 0, u1_arr[k], u0_arr[k])
        rate = eta * (true_signals + u)**2
        p_click = 1.0 - np.exp(-rate * dt)
        clicked = np.random.rand(trials) < p_click
        clicks += clicked.astype(int)
        
    decisions = np.where(clicks % 2 == 0, 1, 0)
    return np.mean(decisions != true_states)

def simulate_bayesian_dolinar(alpha, prior_p0=0.5, nu=0.0, eta=1.0, trials=30000, T=1.0):
    """
    Dolinar receiver with real-time Bayesian continuous drift (tracks 0.5 * exp(-4*alpha^2)).
    """
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    decisions = np.zeros(trials, dtype=int)
    
    for i in range(trials):
        s_alpha = true_alphas[i]
        t = 0.0
        P0 = prior_p0
        
        while t < T:
            f_t = np.exp(-4.0 * alpha**2 * (T - t))
            denom = np.sqrt(max(1.0 - f_t, 1e-12))
            x = 2.0 * P0 - 1.0
            beta = -alpha * (x + denom) / (1.0 + x * denom)
            
            rate_signal = np.abs(s_alpha + beta)**2
            rate_total = eta * rate_signal + nu
            
            if rate_total <= 1e-12:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
                
            delta_t = np.random.exponential(1.0 / rate_total)
            t_next = t + delta_t
            
            if t_next >= T:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
            else:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                P0 = P0 * np.exp(-rate0 * delta_t) / (P0 * np.exp(-rate0 * delta_t) + (1.0 - P0) * np.exp(-rate1 * delta_t))\
                
                P0 = (P0 * rate0) / (P0 * rate0 + (1.0 - P0) * rate1)
                t = t_next
                P0 = np.clip(P0, 1e-12, 1.0 - 1e-12)
                
        decisions[i] = 0 if P0 > 0.5 else 1
        
    return np.mean(decisions != true_states)

def simulate_kennedy(alpha, prior_p0=0.5, nu=0.0, eta=1.0, trials=30000, optimize_disp=False, T=1.0):
    """Simulates static Kennedy detection without dynamic feedback."""
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    
    if optimize_disp:
        beta_mag = get_optimal_kennedy_displacement(alpha, pi0=prior_p0, nu=nu, eta=eta)
        beta = -beta_mag
    else:
        beta = -alpha
        
    rate_true = eta * np.abs(true_alphas + beta)**2 + nu
    clicks = np.random.poisson(rate_true * T)
    decisions = np.where(clicks == 0, 0, 1)
    return np.mean(decisions != true_states)

# Helper function for progress bar
def sweep_ber(alpha_vals, sim_fn, **kwargs):
    desc = kwargs.pop('desc', 'Simulating')
    bers = []
    for a in tqdm(alpha_vals, desc=desc):
        bers.append(sim_fn(a, **kwargs))
    return np.array(bers)

# ==============================================================================
# 3. Execution & Visualization
# ==============================================================================

prior_prob = 0.5          # Prior probability P(H0)
eta = 1.0                 # Detector efficiency
alpha_points = np.linspace(0.1, 1.6, 16)
trials_count = 50000

# 1. Run Simulations
ber_geremia_dolinar = sweep_ber(alpha_points, simulate_geremia_dolinar, eta=eta, pi0=prior_prob, trials=trials_count, desc="Dolinar (Geremia HJB Model)")
ber_bayesian_dolinar = sweep_ber(alpha_points, simulate_bayesian_dolinar, prior_p0=prior_prob, eta=eta, trials=trials_count, desc="Dolinar (Bayesian Adaptive)")
ber_std_kennedy = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, eta=eta, trials=trials_count, optimize_disp=False, desc="Std Kennedy (beta=-alpha)")
ber_opt_kennedy = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, eta=eta, trials=trials_count, optimize_disp=True, desc="Opt Kennedy (beta=-beta_opt)")

# 2. Theoretical Curves
alpha_grid = np.linspace(0.05, 1.8, 200)
helstrom_curve = helstrom_bound(alpha_grid, pi0=prior_prob)
homodyne_curve = homodyne_bound(alpha_grid, pi0=prior_prob)
std_kennedy_curve = standard_kennedy_bound(alpha_grid, pi0=prior_prob)
opt_kennedy_curve = optimized_kennedy_bound(alpha_grid, pi0=prior_prob)

# 3. Plotting
plt.figure(figsize=(11, 7.5))

# Theoretical lines
plt.semilogy(alpha_grid, helstrom_curve, 'k-', linewidth=2.5, label='Helstrom Bound (Quantum Limit)')
plt.semilogy(alpha_grid, homodyne_curve, 'g--', linewidth=2.0, label='Homodyne Detection (SQL)')
plt.semilogy(alpha_grid, std_kennedy_curve, 'c-.', linewidth=2.0, label=r'Standard Kennedy Bound ($\beta=-\alpha$)')
plt.semilogy(alpha_grid, opt_kennedy_curve, color='darkorange', linestyle=':', linewidth=2.2, label=r'Optimized Kennedy Bound ($\beta=-\beta_{opt}$)')

# Simulated data points
plt.semilogy(alpha_points, ber_geremia_dolinar, 'g^', markersize=7, label='Dolinar Receiver (Geremia HJB Model)')
plt.semilogy(alpha_points, ber_bayesian_dolinar, 'mo', markersize=6, label='Dolinar Receiver (Bayesian Adaptive)')
plt.semilogy(alpha_points, ber_std_kennedy, 'c+', markersize=8, markeredgewidth=2, label=r'Standard Kennedy ($\beta=-\alpha$)')
plt.semilogy(alpha_points, ber_opt_kennedy, 'rx', markersize=7, markeredgewidth=1.8, label=r'Optimized Kennedy ($\beta=-\beta_{opt}$)')

plt.title(f'BPSK Quantum Receiver Discrimination: BER vs. Amplitude $\\alpha$ ($P_0={prior_prob}$)', fontsize=12)
plt.xlabel(r'Coherent State Amplitude $\alpha$ (Mean Photon Number $\bar{n}=\alpha^2$)', fontsize=11)
plt.ylabel('Bit Error Rate (BER)', fontsize=11)
plt.ylim(1e-6, 0.8)
plt.xlim(0.1, 1.8)
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.legend(loc='best', fontsize=9.5)
plt.tight_layout()
plt.show()